In [101]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

Dado un sistema de ecuaciones:

$$\left\{
\begin{align*}
    u(x,y) &= 0 ,\\[5pt]
    v(x,y) &= 0 .
\end{align*}
\right.$$

Para hallar la siguiente estimación de los valores de $x$ y $y$, dados por $x_{i+1} = x_i + \Delta x$ y $y_{i+1} = y_i + \Delta y$, se resuelve:

$$
\left[
\begin{array}{cc}
    \left.\frac{\partial u}{\partial x}\right|_{x_i , y_i} & \left.\frac{\partial u}{\partial y}\right|_{x_i , y_i} \\
    \left.\frac{\partial v}{\partial x}\right|_{x_i , y_i} & \left.\frac{\partial v}{\partial y}\right|_{x_i , y_i}
\end{array}
\right] 

\left[
\begin{array}{c}
    \Delta x\\
    \Delta y
\end{array}
\right]
= 
\left[
\begin{array}{c}
    -u(x_i, y_i)\\
    -v(x_i, y_i)
\end{array}
\right]
$$ 

Con el error de tolerancia:

\begin{align*}
    |\epsilon_{rel}|_x = \left| \frac{x_{i+1} - x_i}{x_{i+1}} \right| < \epsilon_s ~~~~~~ y ~~~~~~ |\epsilon_{rel}|_y = \left| \frac{y_{i+1} - y_i}{y_{i+1}} \right| < \epsilon_s
\end{align*}


Con el sistema de ecuaciones:

$$
\left\{
\begin{align*}
    ~(x-x_1)^2 + (y-y1)^2 - d_1 ^2 &= 0 ,\\[5pt]
    ~(x-x_2)^2 + (y-y2)^2 - d_2 ^2 &= 0.
\end{align*}
\right.
$$

In [ ]:
# -------- Parámetros del sistema ----------
x1, y1 = 0, 0 # Mm
x2, y2 = 50, 0 # Mm
d1, d2 = 30, 40 # Mm

# -------- Funciones y Derivadas parciales -------------

def functions(X):

    x, y = X[0], X[1]
    
    u = (x-x1)**2 + (y-y1)**2 - d1**2
    v = (x-x2)**2 + (y-y2)**2 - d2**2 
    
    return np.array([-u, -v])

def partial_diff(X):

    x, y = X[0], X[1]

    du_dx = 2*(x-x1)
    du_dy = 2*(y-y1)

    dv_dx = 2*(x-x2)
    dv_dy = 2*(y-y2)

    return np.array([(du_dx, du_dy), (dv_dx, dv_dy)])

# -------- Parámetros de solución ----------

solution = np.array([20, 20]) # Valores iniciales
tolerance = 0.05 # %

# -------- Eliminación Gaussiana -----------

def gauss_solve(A, b):  # Ax = b
    n = len(b)
    M = np.column_stack((A,b)).astype(float)
    for k in range(n):
        # Pivoteo parcial: buscar el elemento de mayor valor absoluto en una columna dada k
        piv = k + np.argmax(np.abs(M[k:, k]))  
        M[[k, piv]] = M[[piv, k]]

        # Esto es para comprobar si algún pivote resulta siendo muy cercano a cero, complicando los cálculos numéricos para matrices grandes
        # if np.isclose(M[k, k], 0):
        #     raise ValueError("Pivote cero: sistema singular o mal condicionado")

        # Eliminación hacia abajo:
        for i in range(k+1, n):
            factor = M[i, k] / M[k, k]
            for j in range(k, n+1):
                M[i, j] = M[i, j] - factor * M[k, j]

    # Sustitución hacia arriba:
    x = np.zeros(n)
    for i in range(n-1, -1, -1):  # n, n-1, n-2, ..., 1, 0
        suma = 0 
        for j in range(i+1, n):
            suma = suma + M[i, j] * x[j]
        x[i] = (M[i, n] - suma) / M[i, i]
    return x

# -------- Solución Newton-Raphson ---------

history = []
eps = float('inf')
counter = 0

while eps > tolerance:

    partial = partial_diff(solution) 
    func    = functions(solution)
    
    delta = gauss_solve(partial, func)
    solution = solution + delta

    eps = np.max(np.abs(delta / solution)) * 100
    counter += 1

    history.append({
        '$n$': counter,
        '$x_{i+1}$': solution[0],
        '$y_{i+1}$': solution[1],
        r'$\Delta x$': delta[0],
        r'$\Delta y$': delta[1],
        r'$\epsilon_{max}$ $(\%)$': eps 
    })

iteration_table = f"**Tabla de iteraciones para el método Newton-Raphson de sistemas de ecuaciones no lineales:**\n\n" + pd.DataFrame(history).to_markdown(index=False)
display(Markdown(iteration_table))


print(f"El método converge en {counter} iteraciones con [x = {solution[0]:.3f}, y = {solution[1]:.3f}], y residuos {functions(solution)}.")

**Tabla de iteraciones para el método Newton-Raphson de sistemas de ecuaciones no lineales:**

|   $n$ |   $x_{i+1}$ |   $y_{i+1}$ |   $\Delta x$ |   $\Delta y$ |   $\epsilon_{max}$ $(\%)$ |
|------:|------------:|------------:|-------------:|-------------:|--------------------------:|
|     1 |          18 |     24.5    | -2           |    4.5       |                18.3673    |
|     2 |          18 |     24.0051 | -0           |   -0.494898  |                 2.06164   |
|     3 |          18 |     24      |  8.67362e-19 |   -0.0051015 |                 0.0212562 |

El método converge en 3 iteraciones con [x = 18.000, y = 24.000], y residuos [-2.60252883e-05 -2.60252882e-05].
